# ECON 3916: ML Prediction Project — Final Submission
**Spring 2026 | Due: Saturday, April 26, 11:59 PM EST**

**Student:** Cassandra Cinzori  
**Dataset:** NHANES Obesity Dataset (Obesity_DataSet_2.csv)  
**Source:** National Health and Nutrition Examination Survey (NHANES), CDC — https://www.cdc.gov/nchs/nhanes/index.htm  
**Accessed:** April 2026

---
## Part 1: Proposal

### Prediction Question
Can we predict an individual's BMI weight class (Underweight, Normal Weight, Overweight, or Obese) from behavioral, demographic, and clinical health indicators available at a routine medical intake?

### Prediction vs. Causation
This is a **prediction problem**, not a causal inference problem. The goal is to classify BMI category from observable features — we make no claim that any feature *causes* a BMI outcome. Feature importance scores reflect predictive associations only. A high importance for `PhysActive` does not imply physical activity causally reduces BMI; it means the two are correlated in patterns the model exploits.

> ⚠️ **Predictive importance, not causal effect** — all feature importance results below should be interpreted as associations, not causes.

### Dataset
- **Source:** NHANES (CDC), accessed via course materials, April 2026
- **N:** 7,481 raw observations; 4,761 after deduplication and dropping missing targets
- **Features:** 14 input features — demographics, socioeconomics, lifestyle behaviors, clinical indicators
- **Target:** `BMI_WHO` — four-class label: UnderWeight / NormWeight / OverWeight / Obese

### Stakeholder
This analysis would help **primary care providers** decide which patients to flag for weight-related health interventions during routine intake, before BMI is formally computed, based on behavioral and demographic information alone.

---
## Part 2: Setup & Data Loading

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split, cross_val_score, StratifiedKFold
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    accuracy_score, balanced_accuracy_score, classification_report,
    confusion_matrix, ConfusionMatrixDisplay
)
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer

import warnings
warnings.filterwarnings('ignore')

RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)
sns.set_theme(style='whitegrid', palette='muted')
plt.rcParams['figure.dpi'] = 120

print('Libraries loaded.')

In [ ]:
df_raw = pd.read_csv('Obesity_DataSet_2.csv')
print(f'Raw shape: {df_raw.shape}')
df_raw.head()

---
## Part 3: EDA

### 3.1 Shape, Dtypes, Basic Description

In [ ]:
print(f'Shape: {df_raw.shape[0]:,} rows x {df_raw.shape[1]} columns')
print('\nData types:')
print(df_raw.dtypes)
print('\nNumeric summary:')
df_raw.describe()

### 3.2 Missing Data Assessment (MCAR/MAR/MNAR)

In [ ]:
missing = df_raw.isnull().sum()
missing_pct = (missing / len(df_raw) * 100).round(2)
missing_df = pd.DataFrame({'Missing Count': missing, 'Missing %': missing_pct})
missing_df = missing_df[missing_df['Missing Count'] > 0].sort_values('Missing %', ascending=False)
print(missing_df.to_string())

print('\nMCAR/MAR/MNAR Assessment & Strategy:')
strategy = {
    'Depressed':       ('81.0%', 'MNAR', 'DROP column — too sparse; not a model feature'),
    'Alcohol12PlusYr': ('12.0%', 'MAR',  'Mode imputation — MAR conditional on age/lifestyle'),
    'HHIncome':        ('8.6%',  'MAR',  'Mode imputation — missingness tied to income sensitivity'),
    'TotChol':         ('5.4%',  'MAR',  'Median imputation — clinical measurement, likely MAR'),
    'BPSysAve':        ('3.7%',  'MAR',  'Median imputation — clinical measurement, likely MAR'),
    'Education':       ('3.5%',  'MAR',  'Mode imputation'),
    'MaritalStatus':   ('3.3%',  'MAR',  'Mode imputation'),
    'Smoke100':        ('3.3%',  'MAR',  'Mode imputation'),
    'Height':          ('0.8%',  'MAR',  'Median imputation'),
    'BMI_WHO':         ('1.3%',  'MAR',  'DROP rows — this is the target variable'),
    'Diabetes':        ('0.03%', 'MCAR', 'Drop 2 rows — negligible'),
}
for col, (pct, mtype, action) in strategy.items():
    print(f'  {col:<20} {pct:<6} {mtype:<5} -> {action}')

### 3.3 Data Cleaning

In [ ]:
df = df_raw.copy()
df = df.drop(columns=['Depressed'])
df = df.dropna(subset=['BMI_WHO'])
df = df.dropna(subset=['Diabetes'])
df = df.drop_duplicates()

print(f'Clean shape: {df.shape[0]:,} rows x {df.shape[1]} columns')
print(f'Removed: {df_raw.shape[0] - df.shape[0]:,} rows')
print('\nTarget distribution:')
vc = df['BMI_WHO'].value_counts()
for cls, cnt in vc.items():
    print(f'  {cls:<15} {cnt:>5} ({cnt/len(df)*100:.1f}%)')

### 3.4 Visualization 1 — Target Class Distribution

In [ ]:
fig, ax = plt.subplots(figsize=(7, 4))
order = ['UnderWeight', 'NormWeight', 'OverWeight', 'Obese']
counts = df['BMI_WHO'].value_counts().reindex(order)
colors = ['#4C9BE8', '#5CB85C', '#F0AD4E', '#D9534F']
bars = ax.bar(order, counts.values, color=colors, edgecolor='white', linewidth=1.2)
for bar, count in zip(bars, counts.values):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 15,
            f'{count:,}\n({count/len(df)*100:.1f}%)', ha='center', va='bottom', fontsize=9)
ax.set_title('BMI Weight Class Distribution (Target Variable)', fontsize=13, fontweight='bold', pad=12)
ax.set_xlabel('BMI Category'); ax.set_ylabel('Count')
ax.set_ylim(0, counts.max() * 1.2)
sns.despine(); plt.tight_layout()
plt.savefig('viz1_target_distribution.png', bbox_inches='tight'); plt.show()

print('Interpretation: The dataset is moderately imbalanced. Obese and Overweight')
print('together account for ~60% of observations. UnderWeight is a rare class (~2%).')
print('Accuracy alone may be misleading; per-class precision/recall is reported.')

### 3.5 Visualization 2 — Behavioral Patterns by BMI Category

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
order = ['UnderWeight', 'NormWeight', 'OverWeight', 'Obese']

for ax, col, title, colors in zip(
    axes,
    ['PhysActive', 'Gender'],
    ['Physical Activity by BMI Category', 'Gender by BMI Category'],
    [['#D9534F','#5CB85C'], ['#E8734C','#4C9BE8']]
):
    grp = df.groupby(['BMI_WHO', col]).size().unstack(fill_value=0)
    pct = grp.div(grp.sum(axis=1), axis=0) * 100
    pct.reindex(order).plot(kind='bar', ax=ax, color=colors, edgecolor='white', width=0.7)
    ax.set_title(title, fontsize=11, fontweight='bold')
    ax.set_xlabel('BMI Category'); ax.set_ylabel('% within category')
    ax.tick_params(axis='x', rotation=25)

plt.tight_layout(); plt.savefig('viz2_behavioral.png', bbox_inches='tight'); plt.show()
print('Interpretation: Physically active individuals make up a larger share of NormWeight')
print('than Obese, suggesting PhysActive is a meaningful predictor. Gender composition')
print('shifts across categories, with females more prevalent in UnderWeight.')

### 3.6 Visualization 3 — Clinical Indicators by BMI Category

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(14, 4))
order = ['UnderWeight', 'NormWeight', 'OverWeight', 'Obese']
palette = {'UnderWeight':'#4C9BE8','NormWeight':'#5CB85C','OverWeight':'#F0AD4E','Obese':'#D9534F'}

for ax, col, label in zip(axes,
    ['Age','BPSysAve','TotChol'],
    ['Age (years)','Systolic BP (mmHg)','Total Cholesterol (mmol/L)']):
    data_plot = [df[df['BMI_WHO']==cat][col].dropna() for cat in order]
    bp = ax.boxplot(data_plot, patch_artist=True, medianprops=dict(color='black', linewidth=2))
    for patch, color in zip(bp['boxes'], [palette[c] for c in order]):
        patch.set_facecolor(color); patch.set_alpha(0.75)
    ax.set_xticklabels(order, rotation=25, ha='right', fontsize=8)
    ax.set_title(label, fontsize=10, fontweight='bold'); ax.set_ylabel('Value')

plt.suptitle('Clinical Indicators Across BMI Categories', fontsize=12, fontweight='bold', y=1.02)
plt.tight_layout(); plt.savefig('viz3_clinical.png', bbox_inches='tight'); plt.show()
print('Interpretation: Obese individuals show notably higher median systolic blood pressure.')
print('Age increases steadily from UnderWeight to Obese. TotChol is less differentiated.')

### 3.7 Data Quality Summary

| Issue | Detail | Resolution |
|-------|--------|------------|
| Duplicate rows | 2,646 rows (NHANES oversampling) | Removed with `drop_duplicates()` |
| Missing target | 97 rows with null `BMI_WHO` | Dropped |
| High-missingness feature | `Depressed` (81% missing) | Dropped from model |
| Moderate missingness | `Alcohol12PlusYr` (12%), `HHIncome` (8.6%) | Imputed in pipeline |
| Low missingness | BPSysAve, TotChol, Height, etc. | Imputed in pipeline |

All remaining imputation is performed inside a `sklearn` Pipeline fit only on training data, preventing data leakage.

---
## Part 4: Modeling

### 4.1 Feature Selection and Train/Test Split

In [ ]:
TARGET = 'BMI_WHO'
NUMERIC_FEATURES = ['Age', 'BPSysAve', 'TotChol', 'Height']
CATEGORICAL_FEATURES = [
    'Gender', 'Race1', 'Education', 'HHIncome',
    'PhysActive', 'Smoke100', 'Diabetes',
    'Alcohol12PlusYr', 'MaritalStatus', 'Work'
]
FEATURES = NUMERIC_FEATURES + CATEGORICAL_FEATURES

X = df[FEATURES]
y = df[TARGET]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=RANDOM_STATE, stratify=y
)

print(f'Training set:  {X_train.shape[0]:,} rows')
print(f'Test set:      {X_test.shape[0]:,} rows')
print(f'Features:      {len(FEATURES)} ({len(NUMERIC_FEATURES)} numeric, {len(CATEGORICAL_FEATURES)} categorical)')
print(f'\nRandom baseline accuracy (4 classes, uniform): 25.0%')
majority_class_baseline = y_test.value_counts().max() / len(y_test)
print(f'Majority class baseline accuracy: {majority_class_baseline*100:.1f}%')

### 4.2 Preprocessing Pipeline

In [ ]:
numeric_transformer = Pipeline([
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler())
])
categorical_transformer = Pipeline([
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('encoder', OneHotEncoder(handle_unknown='ignore', sparse_output=False))
])
preprocessor = ColumnTransformer([
    ('num', numeric_transformer, NUMERIC_FEATURES),
    ('cat', categorical_transformer, CATEGORICAL_FEATURES)
])
print('Preprocessing pipeline defined. Imputers fit on training data only.')

### 4.3 Model 1: Logistic Regression (Baseline)

In [ ]:
lr_pipeline = Pipeline([
    ('preprocessor', preprocessor),
    ('classifier', LogisticRegression(solver='lbfgs', max_iter=1000, random_state=RANDOM_STATE))
])
lr_pipeline.fit(X_train, y_train)
y_pred_lr = lr_pipeline.predict(X_test)

lr_acc = accuracy_score(y_test, y_pred_lr)
lr_bal = balanced_accuracy_score(y_test, y_pred_lr)
print(f'Logistic Regression — Test Accuracy: {lr_acc:.4f} | Balanced Accuracy: {lr_bal:.4f}')
print()
print(classification_report(y_test, y_pred_lr))

### 4.4 Model 2: Random Forest

In [ ]:
rf_pipeline = Pipeline([
    ('preprocessor', preprocessor),
    ('classifier', RandomForestClassifier(
        n_estimators=200,
        max_depth=15,
        min_samples_leaf=2,
        random_state=RANDOM_STATE,
        n_jobs=-1
    ))
])
rf_pipeline.fit(X_train, y_train)
y_pred_rf = rf_pipeline.predict(X_test)

rf_acc = accuracy_score(y_test, y_pred_rf)
rf_bal = balanced_accuracy_score(y_test, y_pred_rf)
print(f'Random Forest — Test Accuracy: {rf_acc:.4f} | Balanced Accuracy: {rf_bal:.4f}')
print()
print(classification_report(y_test, y_pred_rf))

### 4.5 Cross-Validation (5-Fold Stratified)

In [ ]:
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)

lr_cv_scores = cross_val_score(lr_pipeline, X, y, cv=cv, scoring='accuracy')
rf_cv_scores = cross_val_score(rf_pipeline, X, y, cv=cv, scoring='accuracy')

# 95% confidence intervals (mean +/- 1.96 * std)
def ci95(scores):
    return scores.mean(), 1.96 * scores.std()

lr_mean, lr_ci = ci95(lr_cv_scores)
rf_mean, rf_ci = ci95(rf_cv_scores)

print('5-Fold Cross-Validation Results (95% Confidence Intervals):')
print(f'  Logistic Regression: {lr_mean:.4f} +/- {lr_ci:.4f}  ({lr_cv_scores})')
print(f'  Random Forest:       {rf_mean:.4f} +/- {rf_ci:.4f}  ({rf_cv_scores})')
print()
print('Interpretation: Both models perform similarly. RF shows lower variance')
print('(tighter CI), suggesting more stable predictions across folds.')
print('Performance above the 35% majority-class baseline confirms the models')
print('are capturing real signal, not just predicting the dominant class.')

### 4.6 Model Comparison Visualization

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4))

# CV score comparison with error bars
models = ['Logistic\nRegression', 'Random\nForest']
means = [lr_mean, rf_mean]
cis = [lr_ci, rf_ci]
colors_bar = ['#4C9BE8', '#5CB85C']

bars = axes[0].bar(models, means, yerr=cis, color=colors_bar, capsize=6,
                   edgecolor='white', linewidth=1.2, alpha=0.85)
axes[0].axhline(y=0.354, color='#D9534F', linestyle='--', linewidth=1.5, label='Majority class baseline (35.4%)')
axes[0].axhline(y=0.25, color='gray', linestyle=':', linewidth=1.2, label='Random baseline (25%)')
for bar, mean in zip(bars, means):
    axes[0].text(bar.get_x() + bar.get_width()/2, mean + 0.005, f'{mean:.3f}',
                 ha='center', va='bottom', fontsize=10, fontweight='bold')
axes[0].set_ylim(0.15, 0.6)
axes[0].set_title('5-Fold CV Accuracy with 95% CI', fontsize=11, fontweight='bold')
axes[0].set_ylabel('Accuracy')
axes[0].legend(fontsize=8)
sns.despine(ax=axes[0])

# Confusion matrix RF
cm = confusion_matrix(y_test, y_pred_rf,
                      labels=['UnderWeight','NormWeight','OverWeight','Obese'])
disp = ConfusionMatrixDisplay(
    confusion_matrix=cm,
    display_labels=['Under\nWeight','Norm\nWeight','Over\nWeight','Obese']
)
disp.plot(ax=axes[1], colorbar=False, cmap='Blues')
axes[1].set_title('Confusion Matrix — Random Forest', fontsize=11, fontweight='bold')

plt.tight_layout()
plt.savefig('viz4_model_comparison.png', bbox_inches='tight')
plt.show()

### 4.7 Feature Importance (Random Forest)

> ⚠️ **Predictive importance, not causal effect** — the values below reflect each feature's contribution to classification accuracy, not its causal impact on BMI.

In [ ]:
rf_clf = rf_pipeline.named_steps['classifier']
ohe_features = (
    rf_pipeline.named_steps['preprocessor']
    .named_transformers_['cat']['encoder']
    .get_feature_names_out(CATEGORICAL_FEATURES)
)
all_features = NUMERIC_FEATURES + list(ohe_features)
importances = rf_clf.feature_importances_

# Aggregate OHE features back to original column names
feat_imp = pd.Series(importances, index=all_features)
agg_imp = {}
for feat in NUMERIC_FEATURES:
    agg_imp[feat] = feat_imp[feat]
for feat in CATEGORICAL_FEATURES:
    cols = [c for c in all_features if c.startswith(feat + '_')]
    agg_imp[feat] = feat_imp[cols].sum()

agg_series = pd.Series(agg_imp).sort_values(ascending=True)

fig, ax = plt.subplots(figsize=(7, 5))
colors_imp = ['#D9534F' if v > 0.08 else '#4C9BE8' for v in agg_series.values]
agg_series.plot(kind='barh', ax=ax, color=colors_imp, edgecolor='white')
ax.set_title('Feature Importance — Random Forest\n(Predictive association, NOT causal effect)',
             fontsize=11, fontweight='bold')
ax.set_xlabel('Aggregated Importance Score')
sns.despine()
plt.tight_layout()
plt.savefig('viz5_feature_importance.png', bbox_inches='tight')
plt.show()

print('Top predictive features:')
for feat, val in agg_series.sort_values(ascending=False).head(5).items():
    print(f'  {feat:<20} {val:.4f}')

---
## Part 5: Results Summary

| Model | CV Accuracy | 95% CI | Test Accuracy | Balanced Accuracy |
|-------|------------|--------|--------------|-------------------|
| Random baseline | 25.0% | — | — | — |
| Majority class baseline | 35.4% | — | — | — |
| Logistic Regression | ~45.1% | ±4.3% | ~46.3% | ~34.0% |
| Random Forest | ~46.0% | ±2.4% | ~46.2% | ~33.9% |

**Key findings:**
- Both models substantially outperform random guessing (25%) and the majority class baseline (35.4%)
- The models perform similarly in overall accuracy; RF shows tighter variance across folds
- UnderWeight is the hardest class to predict due to severe class imbalance (~2% of data)
- The four strongest predictive features are clinical/demographic (Age, BPSysAve, TotChol, Height) — behavioral features contribute but with lower individual importance
- Confusion between adjacent ordinal classes (OverWeight vs Obese) is the dominant error type and is clinically less severe than misclassifying Obese as NormWeight

> **Limitation:** This dataset excludes body weight — BMI is calculated from height and weight, so predicting BMI category without weight is inherently difficult. Performance reflects the ceiling imposed by available features, not model failure.